# Grant Funding Statement Triage

Classify funding statements from **cometadata/synthetic-funding-statements** as **High**, **Medium**, or **Low** priority based on metadata completeness.

**Flow:** Load data → Zero-shot baseline (OpenRouter) → Prepare JSONL → Fine-tune (OpenAI) → Evaluate

In [ ]:
import os
import random
import json
from dotenv import load_dotenv
from huggingface_hub import login
from openai import OpenAI
from datasets import load_dataset
from sklearn.metrics import accuracy_score

In [ ]:
# environment

load_dotenv(override=True)
hf_token = os.environ["HF_TOKEN"]
login(hf_token, add_to_git_credential=True)

openrouter_api_key = os.environ.get("OPENROUTER_API_KEY")
openai_api_key = os.environ.get("OPENAI_API_KEY")

if openrouter_api_key:
    print(f"OpenRouter API Key: ...{openrouter_api_key[-3:]}")
else:
    print("OPENROUTER_API_KEY not set")
if openai_api_key:
    print(f"OpenAI API Key: ...{openai_api_key[-4:]}")
else:
    print("OPENAI_API_KEY not set (needed for fine-tuning)")


In [4]:
PRIORITIES = ("High", "Medium", "Low")

def derive_priority(row):
    """Derive priority from funding metadata completeness. High = complete, Low = sparse."""
    score = 0
    funders = row.get("funders") or []
    if not funders:
        return "Low"
    f = funders[0]
    if f.get("funder_name"):
        score += 1
    awards = f.get("awards") or []
    if awards:
        a = awards[0]
        if a.get("award_ids") and len(a["award_ids"]) > 0:
            score += 1
        if a.get("award_title") and len(a["award_title"]) > 0:
            score += 1
        if a.get("funding_scheme") and len(a["funding_scheme"]) > 0:
            score += 1
    if score >= 3:
        return "High"
    if score == 2:
        return "Medium"
    return "Low"


In [ ]:
ds = load_dataset("cometadata/synthetic-funding-statements", split="train")
examples = [
    {"funding_statement": row["funding_statement"], "priority": derive_priority(row)}
    for row in ds
]

random.seed(42)
shuffled = examples.copy()
random.shuffle(shuffled)
n = len(shuffled)
n_train, n_val = int(0.7 * n), int(0.15 * n)
train_data = shuffled[:n_train]
val_data = shuffled[n_train : n_train + n_val]
test_data = shuffled[n_train + n_val :]

print(f"Loaded {len(train_data):,} train, {len(val_data):,} val, {len(test_data):,} test")

In [6]:

openrouter = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=openrouter_api_key)

openai_ft = OpenAI(api_key=openai_api_key) if openai_api_key else None
FRONTIER_MODEL = "openai/gpt-4o-mini"

In [7]:
fine_tune_train = train_data[:500]
fine_tune_validation = val_data[:100]
print(f"Fine-tune: {len(fine_tune_train)} train, {len(fine_tune_validation)} val")

Fine-tune: 500 train, 100 val


In [8]:
len(fine_tune_train)

500

In [10]:
def predict_baseline(statement: str) -> str:
    if not openrouter:
        return "Medium"
    sys_prompt = (
        "You triage grant funding statements. Classify each as exactly one of: High, Medium, Low. "
        "High = complete (funder, award ID, scheme). Medium = partial. Low = sparse. Reply with only that word."
    )
    r = openrouter.chat.completions.create(
        model=FRONTIER_MODEL,
        messages=[
            {"role": "system", "content": sys_prompt},
            {"role": "user", "content": statement},
        ],
        max_tokens=10,
    )
    raw = (r.choices[0].message.content).strip()
    for label in PRIORITIES:
        if label.lower() in raw.lower():
            return label
    return raw or "Medium"

In [11]:
def accuracy(predictor, data):
    correct = sum(1 for ex in data if predictor(ex["funding_statement"]) == ex["priority"])
    return correct / len(data) if data else 0.0

In [ ]:
baseline_acc = accuracy(predict_baseline, test_data[:50])
print(f"Zero-shot baseline accuracy (sample): {baseline_acc:.1%}")

# Step 1 — Prepare JSONL and upload to OpenAI

Prepare training data in JSONL (JSON Lines) format and upload for fine-tuning.

In [13]:
def messages_for(ex):
    """One training example: user = funding statement, assistant = priority."""
    return [
        {"role": "user", "content": ex["funding_statement"]},
        {"role": "assistant", "content": ex["priority"]},
    ]

In [ ]:
messages_for(fine_tune_train[0])

In [15]:
def make_jsonl(items):
    return "\n".join(json.dumps({"messages": messages_for(ex)}) for ex in items)

In [16]:
def write_jsonl(items, filename):
    with open(filename, "w") as f:
        jsonl = make_jsonl(items)
        f.write(jsonl)

In [17]:
os.makedirs("jsonl", exist_ok=True)
write_jsonl(fine_tune_train, "jsonl/fine_tune_train.jsonl")

In [18]:
write_jsonl(fine_tune_validation, "jsonl/fine_tune_validation.jsonl")

In [ ]:
train_file = None
if openai_ft:
    with open("jsonl/fine_tune_train.jsonl", "rb") as f:
        train_file = openai_ft.files.create(file=f, purpose="fine-tune")
    print(f"Uploaded train file: {train_file.id}")
else:
    print("Skipping upload: OPENAI_API_KEY not set")

In [ ]:
train_file

In [ ]:
validation_file = None
if openai_ft:
    with open("jsonl/fine_tune_validation.jsonl", "rb") as f:
        validation_file = openai_ft.files.create(file=f, purpose="fine-tune")
    print(f"Uploaded validation file: {validation_file.id}")

In [ ]:
validation_file

https://platform.openai.com/storage/files/

# Step 2

## And now time to Fine-tune!

In [ ]:
job_id = None
if openai_ft and train_file and validation_file:
    job = openai_ft.fine_tuning.jobs.create(
        training_file=train_file.id,
        validation_file=validation_file.id,
        model="gpt-4.1-nano-2025-04-14",
        seed=42,
        hyperparameters={"n_epochs": 1, "batch_size": 1},
        suffix="grant-triage",
    )
    job_id = job.id
    print(f"Fine-tuning job: {job_id}")
else:
    print("Skipping fine-tune: run upload cells first")

In [25]:
if openai_ft:
    openai_ft.fine_tuning.jobs.list(limit=5)

In [ ]:
if job_id is None and openai_ft:
    job_id = openai_ft.fine_tuning.jobs.list(limit=1).data[0].id
job_id

In [ ]:
job_id

In [ ]:
openai_ft.fine_tuning.jobs.retrieve(job_id) if openai_ft and job_id else None

In [ ]:
openai_ft.fine_tuning.jobs.list_events(fine_tuning_job_id=job_id, limit=10).data if openai_ft and job_id else []

https://platform.openai.com/finetune


# Step 3

Test our fine tuned model

In [39]:
fine_tuned_model_name = openai_ft.fine_tuning.jobs.retrieve(job_id).fine_tuned_model

In [ ]:
fine_tuned_model_name

In [41]:

def test_messages_for(ex):
    return [{"role": "user", "content": ex["funding_statement"]}]

In [ ]:
test_messages_for(test_data[0])

In [43]:
def predict_finetuned(ex):
    """Fine-tuned model predicts priority (requires fine_tuned_model_name)."""
    if not fine_tuned_model_name or not openai_ft:
        return "Medium"
    response = openai_ft.chat.completions.create(
        model=fine_tuned_model_name,
        messages=test_messages_for(ex),
        max_tokens=10,
    )
    
    raw = (response.choices[0].message.content or "").strip()
    for label in PRIORITIES:
        if label.lower() in raw.lower():
            return label
    return raw or "Medium"

In [ ]:
for ex in test_data[:5]:
    pred = predict_finetuned(ex)
    print(f"True: {ex['priority']} | Pred: {pred} | {ex['funding_statement'][:60]}...")

In [ ]:
if fine_tuned_model_name:
    preds = [predict_finetuned(ex) for ex in test_data[:100]]
    truths = [ex["priority"] for ex in test_data[:100]]
    acc = accuracy_score(truths, preds)
    print(f"Fine-tuned accuracy (sample): {acc:.1%}")
else:
    print("Run fine-tuning first to evaluate")